1. [coefficients](#coefficients)

In [32]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
import seaborn as sns
sns.set_palette("pastel")
import plotly.express as px
import matplotlib.pyplot as plt
import math
import numpy as np

from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
%matplotlib inline

In [33]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, make_scorer, confusion_matrix,
)
from sklearn.inspection import permutation_importance

In [34]:
import pandas as pd

# 1. Load data, keep only patients with a known outcome (MCI patients)
df = pd.read_csv('data/plasma_lipidomics.csv')
mci = df[df["Progression to Alzheimer's Disease"].notna()].copy()

# 2. Fill missing numeric values with the column median
numeric_cols = ['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)',
                 'CSF Phosphorylated tau (pg/mL)']
for col in numeric_cols:
    median_val = mci[col].median()
    mci[col] = mci[col].fillna(median_val)

# 3. Fill missing categorical values with the most common value
mode_val = mci['APOE4'].mode()[0]
mci['APOE4'] = mci['APOE4'].fillna(mode_val)

# 4. Convert categorical text columns to numeric (0/1)
mci['Sex'] = (mci['Sex'] == 'Male').astype(int)           # Male=1, Female=0
mci['APOE4'] = (mci['APOE4'] == 'Yes').astype(int)        # carries APOE4 allele=1, no=0

# 5. Set the Target
mci['Target'] = (mci["Progression to Alzheimer's Disease"] == 'Yes').astype(int)

# 5. Final feature set + target
feature_cols = numeric_cols + ['Sex', 'APOE4']
X = mci[feature_cols]
y = mci['Target']

In [35]:
X[:4]

,Age,MMSE,CSF Amyloid (pg/mL),CSF Total tau (pg/mL),CSF Phosphorylated tau (pg/mL),Sex,APOE4
64,69,23,595.0,465.0,75.0,1,0
104,70,27,1845.0,353.0,92.4,1,0
105,73,29,928.0,531.0,176.0,0,0
106,68,23,619.0,477.0,142.0,0,1


## Separate for Training

In [36]:
# Split the data into training and testing sets (stratify to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [37]:
len(X_test)

18

In [38]:
len(X_train)

71

## Models

In [39]:
# Define individual classifiers
models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest':        RandomForestClassifier(n_estimators=300, random_state=42),
    'SVM (RBF)':            Pipeline([('sc', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
    'Gradient Boosting':    GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors':  Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier())]),
    'LDA':                  Pipeline([('sc', StandardScaler()), ('clf', LinearDiscriminantAnalysis())]),
    'Naive Bayes':          Pipeline([('sc', StandardScaler()), ('clf', GaussianNB())]),
}

### Evaluate Classification Models

#### 1. Evaluate by comparing prediction results

In [40]:

# Function to evaluate models. We're concerned about false negatives (missed progressors),
# so we track recall and the F2 score (weights recall twice as heavily as precision)
# alongside accuracy, precision, F1, and ROC-AUC rather than reporting accuracy alone.


def evaluate_model(model,name, X, y):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    # ---- Confusion matrix ----
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
     # ---- Extract metrics ----
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')
    
    return {
        'model': name,
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': recall_score(y, y_pred, zero_division=0),       
        'f2': fbeta_score(y_test, y_pred, beta=2, zero_division=0),
        'roc_auc': roc_auc_score(y, y_proba),
        'false_negatives': fn,
        'false_negative_rate': fnr,
    }

In [41]:
# Evaluate individual models without grid search
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    results.append(evaluate_model(model, name, X_test, y_test))

results_before_gs_df = pd.DataFrame(results)


In [42]:
results = results_before_gs_df.sort_values(by=['f2'], ascending=False)

In [43]:
results

,model,accuracy,precision,recall,f2,roc_auc,false_negatives,false_negative_rate
4,K-Nearest Neighbors,0.777778,0.800000,0.8,0.800000,0.7500,2,0.2
3,Gradient Boosting,0.666667,0.666667,0.8,0.769231,0.7375,2,0.2
1,Random Forest,0.777778,0.875000,0.7,0.729167,0.7250,3,0.3
2,SVM (RBF),0.722222,0.777778,0.7,0.714286,0.7375,3,0.3
6,Naive Bayes,0.666667,0.700000,0.7,0.700000,0.7625,3,0.3
5,LDA,0.611111,0.666667,0.6,0.612245,0.7500,4,0.4
0,Logistic Regression,0.555556,0.600000,0.6,0.600000,0.7750,4,0.4


## GridSearchCV for K_Nearest Neighbors

In [44]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import make_scorer, fbeta_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

#best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
#X_best = X[best_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


pipe = Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier())])

param_grid = [
    {'clf__n_neighbors': [3, 5, 7, 9, 11], 'clf__weights': ['uniform', 'distance'],
     'clf__metric': ['euclidean', 'manhattan']},
    {'clf__n_neighbors': [3, 5, 7, 9, 11], 'clf__weights': ['uniform', 'distance'],
     'clf__metric': ['minkowski'], 'clf__p': [1, 2]},
]

f2_scorer = make_scorer(fbeta_score, beta=2)
scoring_options = {
    'recall': 'recall',
    'f2': f2_scorer,
}

def run_grid_search(scoring_name, scoring):
    grid = GridSearchCV(pipe, param_grid, scoring=scoring, cv=cv, n_jobs=-1)
    
    grid.fit(X, y)
    best_model = grid.best_estimator_

    y_pred = cross_val_predict(best_model, X, y, cv=cv, method='predict')
    y_proba = cross_val_predict(best_model, X, y, cv=cv, method='predict_proba')[:, 1]

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'scoring': scoring_name,
        'best_params': grid.best_params_,
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred),
        'recall': recall_score(y, y_pred, zero_division=0),    
        'roc_auc': roc_auc_score(y, y_proba),
        'false_negatives': int(fn),
        'false_negative_rate': fnr,
        'f2': fbeta_score(y, y_pred, beta=2, zero_division=0),
        'grid':grid
    }

results = [run_grid_search(name, scoring) for name, scoring in scoring_options.items()]
comparison_df = pd.DataFrame(results)
comparison_df['best_params']

0    {'clf__metric': 'euclidean', 'clf__n_neighbors': 7, 'clf__weights': 'distance'}
1     {'clf__metric': 'euclidean', 'clf__n_neighbors': 9, 'clf__weights': 'uniform'}
Name: best_params, dtype: object

In [45]:
comparison_df.columns

Index(['scoring', 'best_params', 'accuracy', 'precision', 'recall', 'roc_auc',
       'false_negatives', 'false_negative_rate', 'f2', 'grid'],
      dtype='str')

In [46]:
comparison_df.iloc[0].grid

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'clf__metric': ['euclidean', 'manhattan'], 'clf__n_neighbors': [3, 5, ...], 'clf__weights': ['uniform', 'distance']}, {'clf__metric': ['minkowski'], 'clf__n_neighbors': [3, 5, ...], 'clf__p': [1, 2], 'clf__weights': ['uniform', 'distance']}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example 

In [47]:
comparison_df['recall']

0    0.702128
1    0.702128
Name: recall, dtype: float64

In [48]:
comparison_df['roc_auc']

0    0.775836
1    0.783435
Name: roc_auc, dtype: float64

<a id="coefficients"></a>
## Coefficients

In [49]:
from sklearn.inspection import permutation_importance

knn_model = models['K-Nearest Neighbors']
knn_model.fit(X_train, y_train)

result = permutation_importance(knn_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

feature_importance_knn_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': result.importances_mean,
    'std': result.importances_std
})
feature_importance_knn_df = feature_importance_knn_df.sort_values(by='importance', ascending=False)
print("\nPermutation Feature Importance from K-Nearest Neighbors:")
print(feature_importance_knn_df)


Permutation Feature Importance from K-Nearest Neighbors:
                          feature  importance       std
3           CSF Total tau (pg/mL)    0.216667  0.067814
2             CSF Amyloid (pg/mL)    0.161111  0.094444
5                             Sex    0.155556  0.095581
0                             Age    0.133333  0.066667
4  CSF Phosphorylated tau (pg/mL)    0.111111  0.065734
6                           APOE4    0.111111  0.049690
1                            MMSE    0.055556  0.043033


## Train the model



In [50]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, roc_auc_score
)
import pandas as pd

best_features = ['CSF Phosphorylated tau (pg/mL)',  'CSF Amyloid (pg/mL)']


X_best = X[best_features]

# ---- Stratified train/test split (80/20), preserves class balance ----
X_train, X_test, y_train, y_test = train_test_split(
    X_best, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train size: {len(X_train)}  ({y_train.sum()} Yes / {len(y_train) - y_train.sum()} No)")
print(f"Test size:  {len(X_test)}  ({y_test.sum()} Yes / {len(y_test) - y_test.sum()} No)")

# ---- Build and fit the final model ----
final_model = Pipeline([
    ('sc', StandardScaler()),
    ('clf', KNeighborsClassifier(metric='euclidean', n_neighbors=7, weights='uniform'))
])

final_model.fit(X_train, y_train)

# ---- Evaluate on the held-out test set ----
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
print(classification_report(y_test, y_pred, target_names=['No progression', 'Progressed'], digits=3))

tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

test_result = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred, zero_division=0),  
    'auc': roc_auc_score(y_test, y_proba),
    'f2': fbeta_score(y_test, y_pred, beta=2, zero_division=0),
    'false_negatives': int(fn),
    'false_negative_rate': fnr,
}
print(test_result)

Train size: 71  (37 Yes / 34 No)
Test size:  18  (10 Yes / 8 No)
             Pred: No  Pred: Yes
Actual: No          5          3
Actual: Yes         3          7
                precision    recall  f1-score   support

No progression      0.625     0.625     0.625         8
    Progressed      0.700     0.700     0.700        10

      accuracy                          0.667        18
     macro avg      0.662     0.662     0.662        18
  weighted avg      0.667     0.667     0.667        18

{'accuracy': 0.6666666666666666, 'precision': 0.7, 'recall': 0.7, 'auc': 0.6125, 'f2': 0.7, 'false_negatives': 3, 'false_negative_rate': np.float64(0.3)}
